# Evaluation pipeline demo

This is a short notebook showing how the evaluation pipeline works WITHOUT having to download the entire dataset and configure the container. Small subsets of each dataset is included in duckdb files in this folder.

Ensure you have Python 3 and `uv`, then just run `uv sync` from the repository root and ensure the notebook is connected to that virtual environment (likely `.venv` by default)

## Datasets
1. [Risk-Based Authentication (RBA)](https://zenodo.org/records/6782156) (Wiefling et al, 2022)
    - synthetic dataset of 31m authentication records from ~3m users with user IDs from an SSO service
    - includes user agent, IP, etc., as well as fields for (a) whether the login originates from a known attack IP address or (b) the login was flagged as an account takeover (ground truth)
    - Licensed by the authors (Wiefling et al.) under a [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

2. [FP-Stalker](https://github.com/Spirals-Team/FPStalker) (Vastel et al., 2018) 
    - 15k record sample from the AmIUnique dataset from 2015-2017. 
    - Each record contains a long term tracking cookie (tying it to other records from the same origin) alongside a full user agent string and other fingerprint attributes we do not use in our ER method (e.g., canvas).
    - Licensed by the authors (Vastel et al.) under the [GNU-AGPL 3.0 License](https://github.com/Spirals-Team/FPStalker/blob/master/LICENSE)



The datasets provided in this demo are small subsets of the above datasets just to show how this works at a high level:
1. RBA subset (3.2k rows): Selected only only "User IDs" that have at least one row flagged as "Is Account Takeover"
2. FP Stalker Subset (8.6k rows): Selected only tracking cookie IDs that were live in June 2016. Also removed some columns that aren't useful to us.


In [ ]:
import duckdb
import pandas as pd

with duckdb.connect('./trunc_rba.duckdb') as rba_conn:
    rba_df_orig = rba_conn.execute("SELECT * FROM imported_data").df()

print("Length of RBA truncated dataset:", rba_df_orig.shape[0])
display(rba_df_orig.head())

In [ ]:
with duckdb.connect('./trunc_fp_stalker.duckdb') as fp_conn:
    fp_df_orig = fp_conn.execute("SELECT * FROM imported_data").df() 

print("Length of FP Stalker truncated dataset:", fp_df_orig.shape[0])
display(fp_df_orig.head())

In [ ]:
import sys
from pathlib import Path
import tqdm

_root = Path.cwd().parent.parent
_python_core = str(_root / "python_core")
if _python_core not in sys.path:
    sys.path.append(_python_core)
if str(_root) not in sys.path:
    sys.path.append(str(_root))

from python_core.field_normalization.user_agent import UserAgentParser
from python_core.field_normalization.device import normalize_device_fields

In [ ]:
ua_parser = UserAgentParser()

def normalize_data(df: pd.DataFrame, 
                   user_agent_col_name: str, 
                   timestamp_col_name: str,
                   other_cols_to_keep: list = []) -> pd.DataFrame:
    dicts = []
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        d = {"user_agent_original": row.get(user_agent_col_name, "")}
        d = ua_parser.parse(d) # returns dict
        d = normalize_device_fields(d)
        d = {k: v for k, v in d.items() if k.startswith("norm__")}
        d['timestamp'] = row.get(timestamp_col_name, "")
        dicts.append(d)
    
    new_columns = pd.DataFrame(dicts, index=df.index)
    normalized_cols = [c for c in new_columns.columns if c.startswith("norm__")]
    
    if isinstance(other_cols_to_keep, list):
        other_cols_to_keep = [c for c in other_cols_to_keep if c in df.columns]
    else:
        other_cols_to_keep = []
    
    new_df = df[other_cols_to_keep + [user_agent_col_name]]
    new_df = pd.concat([
        new_columns[["timestamp"] + normalized_cols], 
        new_df], axis=1)
    new_df.rename(columns={c: f"attr__{c}" for c in normalized_cols}, inplace=True)
    return new_df

In [ ]:
s = "Mozilla/5.0  (iPhone; CPU iPhone OS 11_2_6 like Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko Version/4.0 Chrome/85.0.4183.127 Mobile Safari/537.36Snapchat11.1.7.81 (LYA-L29; Android 10#10.1.0.288C432#29; gzip"
s = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/69.0.3497.17.19.92 Safari/537.36"
s = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.1 Safari/605.1.15"
s = "Mozilla/5.0  (iPad; CPU OS 7_1 like Mac OS X) AppleWebKit/533.1 (KHTML, like Gecko Version/4.0 Mobile Safari/533.1 variation/277457"
ua_parser.parse({"user_agent_original": s})

In [ ]:
rba_df_orig = rba_df_orig[rba_df_orig['Login Successful'] == True]
norm_rba_df = normalize_data(rba_df_orig,
                             user_agent_col_name='User Agent String', 
                             timestamp_col_name='Login Timestamp',
                             other_cols_to_keep=['index', 'User ID', 'Login Successful', 'Is Account Takeover', 'Is Attack IP', 'Browser Name and Version', 'OS Name and Version'])                           

norm_rba_df.rename(columns={'index': 'id'},  inplace=True)  # "id" has to be the unique id of the row
norm_rba_df.convert_dtypes()
norm_rba_df

In [ ]:
norm_fp_df = normalize_data(fp_df_orig,
                            user_agent_col_name='userAgentHttp', 
                            timestamp_col_name='creationDate',
                            other_cols_to_keep=['id', 'counter', 'osDetailed', 'browserDetailed'])

norm_fp_df.rename(columns={'id': 'tracking_id'},  inplace=True)  # "id" has to be the unique id of the row
norm_fp_df.rename(columns={'counter': 'id'}, inplace=True)

norm_fp_df.convert_dtypes()
norm_fp_df

# (A) RBA Dataset

We evaluate our device grouping heuristic using the dataset from:
> Stephan Wiefling, Paul René Jørgensen, Sigurd Thunem, and Luigi Lo Iacono. 2022. **Pump Up Password Security! Evaluating and Enhancing Risk-Based Authentication on a Real-World Large-Scale Online Service**. *ACM Transactions on Privacy and Security (TOPS)*. [DOI: 10.1145/3549079](https://doi.org/10.1145/3549079) | [Zenodo Dataset](https://doi.org/10.5281/zenodo.6782155)

The dataset contains synthetic values for privacy. Each record includes:
- User account ID (`User ID`)
- User agent and metadata fields (`User Agent String`)
- Account takeover flag (`Is Account Takeover` - ground-truth compromise)
- Known attack IP flag (`Is Attack IP` - ground-truth indicator of compromise)

For all user accounts with at least one flagged account takeover, we group their logins into device instances and evaluate:
1. Do we accidentally group unflagged logins with flagged logins?
2. Are these errors caused by spoofed user agents matching unflagged ones?

In [ ]:
from python_core.device_grouping2 import client_os_upgrades, profiles
from python_core.device_grouping2.instances import DeviceInstanceGraph
import datetime
import json

### (A1) Compute "instances" from records

We use `device_grouping2` modules to first compute client upgrade edges between login records, then compute the connected subgraphs (representing device instances).

`rba_calculate_instances` iteratively adds these instance IDs (e.g., `{User ID}_inst_{i}`) to the DataFrame.

It takes `max_days_client`, the maximum time gap (in days) allowed between consecutive logins on a client.

In [ ]:
def rba_calculate_instances(df: pd.DataFrame, max_days_client: int):
    prefix = f"run_{max_days_client}d"
    df[f'{prefix}_predicted_instance_id'] = None
    
    for user_id in map(int, df['User ID'].dropna().unique()):
        user_df = df[df['User ID'] == user_id]
        upgrade_edges = client_os_upgrades.get_edges(user_df, max_days_client=max_days_client, run_pass2=False)
        graph = DeviceInstanceGraph(user_df, pd.DataFrame(upgrade_edges))

        for i, inst in enumerate(graph.get_instances()):
            df.loc[inst.df.index, f'{prefix}_predicted_instance_id'] = f"{user_id}_inst_{i}"

When a user logs in from a client, the heuristic will only group consecutive logins into the same device instance if the time gap between them is less than `max_days_client`.

We iterate over multiple `max_days_client` to find the optimal threshold.

In [ ]:
start_time = datetime.datetime.now().isoformat()
max_days_client_values = [7, 14, 30, 60, 90]

for days in max_days_client_values:
    rba_calculate_instances(norm_rba_df, max_days_client=days)

### (A2) Compute summary statistics

Recall that individual records may be flagged as either `Is Account Takeover` or `Is Attack IP`. For a given flag, a record is flagged (value of `1`) or unflagged (value of `0`).

For every calculated instance (subgraph containing multiple individual records), we classify its type separately for each flag (resulting in two columns: `takeover_instance_type` and `attack_ip_instance_type`):
- `only_unflagged`: The instance contains only unflagged records  --> Good/Homogeneous
- `only_flagged`: The instance contains only flagged records  --> Good/Homogeneous
- `mixed`: Contains both flagged and unflagged records for that flag --> Contaminated

Our goal is for instances to be homogeneous, either `only_unflagged` or `only_flagged`. For `mixed` instances (erroneous), we calculate whether they are:
- `containing_identical_ua`: At least one flagged record has the same, identical User Agent string as an unflagged record within that instance. This is an unavoidable error caused by data limitations (e.g., an attacker spoofing a legitimate user's exact User Agent string).
- `containing_heuristic_error`: At least one flagged record has a User Agent string that does not match any of the unflagged records in the instance. This means the client upgrade heuristic incorrectly drew upgrade edges between different User Agents (avoidable algorithm error).

In [ ]:
summary = {
    "run": {
        "description": "Truncated RBA dataset parameter sweep (client window)",
        "start_time": start_time,
        "max_days_client_values": max_days_client_values
    },
    "dataset": {
        "num_records": len(norm_rba_df),
        "num_users": len(norm_rba_df['User ID'].unique())
    },
    "results": {}
}

In [ ]:
for days in max_days_client_values: # for every run
    run_prefix = f"run_{days}d"
    gp = norm_rba_df.groupby(f'{run_prefix}_predicted_instance_id')
    sizes = gp.size()
    
    run_summary = {
        "num_instances": {
            "total": len(gp),
            "singletons": int((sizes == 1).sum()),
            "median_size": int(sizes.median()),
            "mean_size": float(sizes.mean()),
            "max_size": int(sizes.max()),
        }
    }
    
    for key, col in [
        ("takeover", "Is Account Takeover"),
        ("attack_ip", "Is Attack IP")
]:
        is_flagged = norm_rba_df[col] == 1
        is_unflagged = norm_rba_df[col] == 0
        
        instance_has_flagged = is_flagged.groupby(norm_rba_df[f'{run_prefix}_predicted_instance_id']).any()
        instance_has_unflagged = is_unflagged.groupby(norm_rba_df[f'{run_prefix}_predicted_instance_id']).any()
        
        inst_type = pd.Series("only_unflagged", index=instance_has_flagged.index)
        inst_type[instance_has_flagged & instance_has_unflagged] = "mixed"
        inst_type[instance_has_flagged & ~instance_has_unflagged] = "only_flagged"
        norm_rba_df[f'{run_prefix}_{key}_instance_type'] = norm_rba_df[f'{run_prefix}_predicted_instance_id'].map(inst_type)
        
        # B-cubed calculation
        inst_size = norm_rba_df[f'{run_prefix}_predicted_instance_id'].map(sizes)
        same_flag_per_inst = is_flagged.groupby(norm_rba_df[f'{run_prefix}_predicted_instance_id']).transform("sum").where(is_flagged, is_unflagged.groupby(norm_rba_df[f'{run_prefix}_predicted_instance_id']).transform("sum"))
        total_flag_per_user = is_flagged.groupby(norm_rba_df["User ID"]).transform("sum").where(is_flagged, is_unflagged.groupby(norm_rba_df["User ID"]).transform("sum"))
        b3_precision = float((same_flag_per_inst / inst_size).mean())
        b3_recall = float((same_flag_per_inst / total_flag_per_user).mean())
        
        mixed_ids = instance_has_flagged[instance_has_flagged & instance_has_unflagged].index
        mixed_df = norm_rba_df[norm_rba_df[f'{run_prefix}_predicted_instance_id'].isin(mixed_ids)]
        
        mixed_identical_ua = 0
        mixed_heuristic_error = 0
        
        if not mixed_df.empty:
            for _, inst_df in mixed_df.groupby(f'{run_prefix}_predicted_instance_id'):
                unflagged_uas = set(inst_df[inst_df[col] == 0]['User Agent String'].dropna())
                flagged_uas = set(inst_df[inst_df[col] == 1]['User Agent String'].dropna())
                
                if flagged_uas & unflagged_uas:
                    mixed_identical_ua += 1
                if flagged_uas - unflagged_uas:
                    mixed_heuristic_error += 1
                    
        run_summary[key] = {
            "bcubed_precision": b3_precision,
            "bcubed_recall": b3_recall,
            "num_records": {
                "total": len(norm_rba_df),
                "flagged": int(is_flagged.sum()),
                "unflagged": int(is_unflagged.sum())
            },
            "num_instances": {
                "total": len(gp),
                "containing_only_flagged": int((instance_has_flagged & ~instance_has_unflagged).sum()),
                "containing_only_unflagged": int((~instance_has_flagged & instance_has_unflagged).sum()),
                "containing_mixed": {
                    "total": len(mixed_ids),
                    "containing_identical_ua": mixed_identical_ua,
                    "containing_heuristic_error": mixed_heuristic_error
                }
            }
        }
        
    summary["results"][run_prefix] = run_summary

Finally, look at the results (for `Is Account Takeover` only, since this is our main evaluation target):
* `max_days_client`: The client login threshold (in days) being evaluated.
* `Num Instances`: The total number of device instances generated across all users.
* `Num Instances (Flagged)`: The number of device instances containing at least one account takeover login (the vulnerable population).
* `Num Instances w/ Heuristic Error`: The number of takeover-containing instances that were incorrectly grouped by the algorithm (avoidable errors).
* `BCubed Precision`: How clean our device instances are. A score of `1.0` means no account takeover logins were mixed with unflagged logins.
* `BCubed Recall`: How complete our device instances are. A score of `1.0` means all logins belonging to the same entity (user or attacker) were successfully grouped together.
* `BCubed F1`: The balanced F1 score combining precision and recall.
* `BCubed F0.5`: The precision-weighted F-score, weighting precision twice as heavily as recall to prioritize security.

In [ ]:
pd.DataFrame([{
    "max_days_client": int(k.split("_")[1][:-1]),
    "Num Instances": v["num_instances"]["total"],
    "Num Instances (Flagged)": v["takeover"]["num_instances"]["containing_only_flagged"] + v["takeover"]["num_instances"]["containing_mixed"]["total"],
    "Num Instances w/ Heuristic Error": v["takeover"]["num_instances"]["containing_mixed"]["containing_heuristic_error"],
    "BCubed Precision": v["takeover"]["bcubed_precision"],
    "BCubed Recall": v["takeover"]["bcubed_recall"],
    "BCubed F1": 2 * (v["takeover"]["bcubed_precision"] * v["takeover"]["bcubed_recall"]) / (v["takeover"]["bcubed_precision"] + v["takeover"]["bcubed_recall"]),
    "BCubed F0.5": (1 + 0.5**2) * (v["takeover"]["bcubed_precision"] * v["takeover"]["bcubed_recall"]) / ((0.5**2 * v["takeover"]["bcubed_precision"]) + v["takeover"]["bcubed_recall"]),
} for k, v in summary["results"].items()])

We export the consolidated summary metrics to JSON, and the login records (with all run predictions) to a single CSV.

In [ ]:
_start_time = start_time.replace(":", "-").replace(".", "-")
dirpath =  Path("runs") / f"rba_sweep_{_start_time}"
dirpath.mkdir(parents=True, exist_ok=True)

summary["run"]["end_time"] = datetime.datetime.now().isoformat()
summary["run"]["duration"] = str(datetime.datetime.fromisoformat(summary["run"]["end_time"]) - datetime.datetime.fromisoformat(start_time))

with open(dirpath / "summary.json", "w") as f:
    json.dump(summary, f, indent=4)

df_to_save = norm_rba_df.drop(columns=['Login Successful', 'Browser Name and Version', 'OS Name and Version'], errors='ignore')

run_cols = [c for c in df_to_save.columns if c.startswith('run_')]
other_cols = [c for c in df_to_save.columns if c not in run_cols and c != 'User ID']

df_to_save[['User ID'] + run_cols + other_cols].to_csv(dirpath / "logins_with_predictions.csv", index=False)


# (B) FP Stalker Dataset

We evaluate the client-grouping heuristic on the FP-Stalker dataset. Since FP-Stalker contains logins labeled with ground-truth device cookies (`tracking_id`), we simulate a user owning multiple devices by randomly mixing $K$ different tracking IDs under a single synthetic `Synth User ID`.

Because the dataset subset is restricted to **June 2016**, it naturally controls for temporal concurrency (meaning all mixed devices were active during the same month).

### (B1) Compute "instances" from mixed devices

We define a helper function `fp_calculate_instances` that groups logins by our synthetic `Synth User ID`. 

For each synthetic user, it calculates client upgrade edges (using the default 30-day client window and no OS upgrades) and groups them into device instances.

In [ ]:
def run_prefix(k: int, max_days_client: int) -> str:
    return f"mix_{k}way_run_{max_days_client}d"

def synth_user_col(k: int) -> str:
    return f"mix_{k}way_synth_user_id"

def fp_calculate_instances(df: pd.DataFrame, max_days_client: int, k: int):
    prefix = run_prefix(k, max_days_client)
    df[f'{prefix}_predicted_instance_id'] = None
    
    for user_id in df[synth_user_col(k)].dropna().unique():
        user_df = df[df[synth_user_col(k)] == user_id]
        upgrade_edges = client_os_upgrades.get_edges(user_df, max_days_client=max_days_client)
        graph = DeviceInstanceGraph(user_df, pd.DataFrame(upgrade_edges))

        for i, inst in enumerate(graph.get_instances()):
            df.loc[inst.df.index, f'{prefix}_predicted_instance_id'] = f"{user_id}_inst_{i}"

We run a parameter sweep over different mixing levels $K \in [2, 3, 4, 5]$ (number of physical devices mixed under a single synthetic user). 

For each mixing level, we:
1. **Shuffle and group** the unique `tracking_id`s into synthetic users of size $K$, assigning the mapping to `Synth User ID`.
2. **Run the client-grouping heuristic** using the synthetic `Synth User ID` as the grouping key.
3. **Calculate B-cubed metrics** at the record level using fast $O(N)$ grouping:
   * `same_tracking_id_per_inst`: Count of records in the same predicted instance that share the same physical device (`tracking_id`).
   * `inst_size`: Total size of the predicted instance.
   * `total_tracking_id`: Total number of records belonging to that physical device (`tracking_id`).
   * $\text{bcubed\_precision} = \text{mean}(\text{same\_tracking\_id\_per\_inst} / \text{inst\_size})$
   * $\text{bcubed\_recall} = \text{mean}(\text{same\_tracking\_id\_per\_inst} / \text{total\_tracking\_id})$


In [ ]:
import numpy as np

_fp_start_time = datetime.datetime.now().isoformat()
ks = [2, 3, 4, 5]
unique_trackID = norm_fp_df['tracking_id'].dropna().unique()

np.random.seed(42)
fp_summary = {
    "run": {
        "description": "Truncated FP-Stalker dataset parameter sweep (device mixing)",
        "start_time": "",
        "K_values_iterated": ks
    },
    "dataset": {
        "num_records": len(norm_fp_df),
        "num_devices": len(unique_trackID)
    },
    "results": {}
}

In [ ]:
fp_start_time = datetime.datetime.now().isoformat()
fp_summary["run"]["start_time"] = fp_start_time

for k in ks:
    np.random.shuffle(unique_trackID)
    trackID_to_user = pd.Series(np.arange(len(unique_trackID)) // k, index=unique_trackID)
    norm_fp_df[synth_user_col(k)] = norm_fp_df['tracking_id'].map(trackID_to_user)

    print(f"Running for k={k}... with {len(unique_trackID)} unique devices, resulting in {len(trackID_to_user.unique())} synthetic users.")
    for max_days in max_days_client_values: # from the RBA runs
        print(f"  Running for max_days={max_days}...")
        fp_calculate_instances(norm_fp_df, max_days_client=max_days, k=k)
    
        prefix = run_prefix(k, max_days)
        pred_col = f"{prefix}_predicted_instance_id"
        gp = norm_fp_df.groupby(pred_col)

        same_tracking_id_per_inst = norm_fp_df.groupby([pred_col, "tracking_id"]).transform("size")
        precision = float((same_tracking_id_per_inst / norm_fp_df.groupby(pred_col).transform("size")).mean())
        recall = float((same_tracking_id_per_inst / norm_fp_df.groupby("tracking_id").transform("size")).mean())


        fp_summary["results"][prefix] = {
            "bcubed_precision": precision,
            "bcubed_recall": recall,
            "num_instances": {
                "total": norm_fp_df[pred_col].nunique()
            }
        }

### (B2) Compute summary statistics

The table below evaluates client-grouping accuracy against the number of mixed physical devices ($K$) per synthetic user:
* `mix_k_devices`: The number of physical devices mixed per synthetic user.
* `Num Instances`: The total number of device instances generated.
* `BCubed Precision`: How clean our device instances are (1.0 means no mixed devices).
* `BCubed Recall`: How complete our device instances are (1.0 means all logins from a device were grouped together).
* `BCubed F1`: The balanced F1 score combining precision and recall.
* `BCubed F0.5`: The precision-weighted F-score, weighting precision twice as heavily as recall to prioritize security.

In [ ]:
df_fp_summary = pd.DataFrame([{
    "mix_k_devices": int(k.split("_")[1].replace("way", "")),
    "max_days_client": int(k.split("_")[3].replace("d", "")),
    "Num Instances": v["num_instances"]["total"],
    "BCubed Precision": v["bcubed_precision"],
    "BCubed Recall": v["bcubed_recall"],
} for k, v in fp_summary["results"].items()])

p, r = df_fp_summary["BCubed Precision"], df_fp_summary["BCubed Recall"]
df_fp_summary["BCubed F1"] = 2 * (p * r) / (p + r)
df_fp_summary["BCubed F0.5"] = 1.25 * (p * r) / (0.25 * p + r)

display(df_fp_summary.sort_values(["mix_k_devices", "max_days_client"]))

We export the consolidated FP-Stalker summary metrics to JSON, and the login records (with all run predictions) to a single CSV.

In [ ]:
_fp_start_time = fp_start_time.replace(":", "-").replace(".", "-")
dirpath = Path("runs") / f"fp_stalker_{_fp_start_time}"
dirpath.mkdir(parents=True, exist_ok=True)

fp_summary["run"]["end_time"] = datetime.datetime.now().isoformat()
fp_summary["run"]["duration"] = str(datetime.datetime.fromisoformat(fp_summary["run"]["end_time"]) - datetime.datetime.fromisoformat(fp_start_time))
    

with open(dirpath / "summary.json", "w") as f:
  json.dump(fp_summary, f, indent=4)
  
df_to_save = norm_fp_df.drop(columns=['osDetailed', 'browserDetailed'], errors='ignore')
run_cols = [c for c in df_to_save.columns if c.startswith('mix_')]
other_cols = [c for c in df_to_save.columns if c not in run_cols and c not in ['Synth User ID', 'tracking_id']]
  
df_to_save[['User ID'] + run_cols + other_cols].to_csv(dirpath / "logins_with_predictions.csv", index=False)